In [1]:
# Фикс конфликта protobuf/tf_keras (AttributeError GetPrototype)
!pip install protobuf==3.20.3 --force-reinstall -q
!pip install transformers==4.44.0 datasets==2.20.0 torch scikit-learn pandas numpy tqdm -q

import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, bu

In [2]:
# import os
import numpy as np
import pandas as pd
import json

from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, roc_auc_score
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

2025-12-10 19:03:29.068717: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765393409.437655      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765393409.537347      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
# Загрузка датасета

INPUT_DIR = '/kaggle/input/you-are-bot/'

# with open(f'{INPUT_DIR}train.json', 'r') as f:
#     train_data = json.load(f)
# 
# with open(f'{INPUT_DIR}test.json', 'r') as f:
#     test_data = json.load(f)
    
with open(f'{INPUT_DIR}train.json', 'r', encoding='utf-8') as f:
    train_data = json.load(f)

with open(f'{INPUT_DIR}test.json', 'r', encoding='utf-8') as f:
    test_data = json.load(f)

# ytrain (конвертируем participant_index в строку)
ytrain = pd.read_csv(f'{INPUT_DIR}ytrain.csv')
ytrain['participant_index'] = ytrain['participant_index'].astype(str)

# ytest
ytest = pd.read_csv(f'{INPUT_DIR}ytest.csv')
ytest['participant_index'] = ytest['participant_index'].astype(str)

sample_sub = pd.read_csv(f'{INPUT_DIR}sample_submission.csv')

print(f"Train dialogs: {len(train_data)}, Labels: {len(ytrain)}")
print(f"Test dialogs: {len(test_data)}, IDs: {len(ytest)}")

Train dialogs: 786, Labels: 1572
Test dialogs: 338, IDs: 676


In [4]:
# Проверка train_data: Основные статистики и примеры
# 1. Общая инфа
num_dialogs = len(train_data)
print(f"Количество диалогов в train: {num_dialogs}")  # Ожидаемо ~50,000

# 2. Первый диалог: Полная структура (dict с list сообщений)
first_dialog_id = list(train_data.keys())[0]  # Берем первый ключ (dialog_id)
first_dialog = train_data[first_dialog_id]
print(f"\nПервый диалог (ID: {first_dialog_id}):")
print(f"Тип: {type(first_dialog)}")  # <class 'list'>
print(f"Количество сообщений: {len(first_dialog)}")  # ~5-20
print("Структура первого сообщения:", first_dialog[0] if first_dialog else "Пустой!")  # Пример: {'text': '...', 'message': 1, 'participant_index': 0}

# 3. Тексты первых 3 сообщений (чтобы увидеть реальный контент диалога)
print("\nПервые 3 сообщения из первого диалога:")
for i, msg in enumerate(first_dialog[:3]):
    print(f"  Сообщение {i+1} (участник {msg['participant_index']}): {msg['text'][:100]}...")  # Обрезаем до 100 символов для читаемости

# 4. Краткая статистика по всем диалогам
dialog_lengths = [len(dialog) for dialog in train_data.values()]
avg_length = np.mean(dialog_lengths)
print(f"\nСтатистика: Средняя длина диалога: {avg_length:.1f} сообщений")

# Баланс participant_index (0 vs 1) — проверим на паре диалогов
participant_counts = {'0': 0, '1': 0}
for dialog in list(train_data.values())[:100]:  # Проверяем первые 100 для скорости
    for msg in dialog:
        participant_counts[str(msg['participant_index'])] += 1
print(f"Баланс участников (первые 100 диалогов): 0={participant_counts['0']}, 1={participant_counts['1']}")

# Опционально: Сохрани пример в файл для скачивания (если нужно)
with open('example_dialog.json', 'w') as f:
    json.dump({first_dialog_id: first_dialog}, f, indent=2)
print("\nПример диалога сохранён в 'example_dialog.json' (можно скачать из Output).")

Количество диалогов в train: 786

Первый диалог (ID: dae9e2dae9f840549764f8d9bbbb80f0):
Тип: <class 'list'>
Количество сообщений: 5
Структура первого сообщения: {'message': 0, 'text': 'Hello!', 'participant_index': '0'}

Первые 3 сообщения из первого диалога:
  Сообщение 1 (участник 0): Hello!...
  Сообщение 2 (участник 1): Как дела?...
  Сообщение 3 (участник 0): Отлично! А твои?...

Статистика: Средняя длина диалога: 8.1 сообщений
Баланс участников (первые 100 диалогов): 0=423, 1=376

Пример диалога сохранён в 'example_dialog.json' (можно скачать из Output).


In [5]:
# ytrain.head()


# Проверка типов в ytrain (после astype)
print("Типы колонок в ytrain:")
print(ytrain.dtypes)  # Ожидаемо: dialog_id object (str), participant_index object (str), is_bot int64

print("\nПервые 5 значений participant_index:")
print(ytrain['participant_index'].head())  # '0', '0', ... (str)

print("\nПример: Тип первого значения:", type(ytrain['participant_index'].iloc[0]))  # <class 'str'>

# Тест матчинга (с твоим sample)
sample_idx = ytrain['participant_index'].iloc[0]  # '0'
print(f"Сравнение с JSON-str: '{sample_idx}' == '0' → {sample_idx == '0'}")  # True
print(f"Сравнение с int: '{sample_idx}' == 0 → {sample_idx == 0}")  # False (если бы не фикс)

Типы колонок в ytrain:
dialog_id            object
participant_index    object
is_bot                int64
dtype: object

Первые 5 значений participant_index:
0    0
1    0
2    0
3    0
4    0
Name: participant_index, dtype: object

Пример: Тип первого значения: <class 'str'>
Сравнение с JSON-str: '0' == '0' → True
Сравнение с int: '0' == 0 → False


In [6]:
# Предобработка: Функция для извлечения текста по participant_index

def extract_participant_text(dialog, participant_index):
    """Извлекает текст для participant_index (str: '0' или '1')."""
    messages = [msg['text'] for msg in dialog if msg['participant_index'] == participant_index]
    return ' '.join(messages).strip()

# Train
train_texts = []
train_labels = []

for dialog_id, dialog in tqdm(train_data.items(), desc="Processing train"):
    matching_rows = ytrain[ytrain['dialog_id'] == dialog_id]  # dialog_id str — матчит
    for _, row in matching_rows.iterrows():
        idx = row['participant_index']  # Теперь str
        text = extract_participant_text(dialog, idx)
        if text:
            train_texts.append(text)
            train_labels.append(int(row['is_bot']))

train_df = pd.DataFrame({'text': train_texts, 'label': train_labels})
print(f"Processed train samples: {len(train_df)} (из {len(train_data)} диалогов)")

# Проверка
if len(train_df) == 0:
    print("0 samples! Видимо ошибка с типами в ytrain (participant_index str).")
else:
    print(f"Баланс: {train_df['label'].value_counts().to_dict()}")
    # Пример: Первый текст + label
    print("\nПример из train_df:")
    print(f"Text: {train_df.iloc[0]['text'][:100]}...")  # Обрезаем
    print(f"Label: {train_df.iloc[0]['label']}")

# Test (аналогично)
test_texts = []
test_ids = []

for dialog_id, dialog in tqdm(test_data.items(), desc="Processing test"):
    matching_rows = ytest[ytest['dialog_id'] == dialog_id]
    for _, row in matching_rows.iterrows():
        idx = row['participant_index']  # str
        text = extract_participant_text(dialog, idx)
        if text:
            test_texts.append(text)
            test_ids.append(row['ID'])  # Уже готово

test_df = pd.DataFrame({'text': test_texts, 'ID': test_ids})
print(f"Processed test samples: {len(test_df)}")

Processing train: 100%|██████████| 786/786 [00:00<00:00, 1568.01it/s]


Processed train samples: 1572 (из 786 диалогов)
Баланс: {0: 1033, 1: 539}

Пример из train_df:
Text: Hello! Отлично! А твои? Расскажи теорему...
Label: 0


Processing test: 100%|██████████| 338/338 [00:00<00:00, 1997.85it/s]

Processed test samples: 676


In [7]:
# Split на train/val и токенизация

# Split (80/20)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['text'].tolist(), train_df['label'].tolist(), test_size=0.2, random_state=42, stratify=train_df['label']
)

# Токенизатор
# tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-multilingual-cased')

class DialogDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = texts
        self.labels = labels
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=512, return_tensors='pt')
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

# Датасеты
train_dataset = DialogDataset(train_texts, train_labels)
val_dataset = DialogDataset(val_texts, val_labels)
test_dataset = DialogDataset(test_df['text'].tolist())  # Без labels

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

In [8]:
# Модель и тренировка (Trainer API)


# Модель
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-multilingual-cased', num_labels=2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Args: Отключить WandB явно
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to="none",  # ← Явно none (no WandB/TensorBoard)
    logging_strategy="no"  # ← Нет логов (опционально, для тишины)
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    probs = np.exp(predictions) / np.sum(np.exp(predictions), axis=1)[:, None]
    return {
        'log_loss': log_loss(labels, probs[:, 1]),
        'auc': roc_auc_score(labels, probs[:, 1])
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# Тренировка (теперь без hang)
trainer.train()
print("Training completed!")

model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Log Loss,Auc,Runtime,Samples Per Second,Steps Per Second
1,No log,0.525917,0.525579,0.777688,3.422000,92.051000,5.845000
2,No log,0.534139,0.534058,0.788916,3.662300,86.011000,5.461000
3,No log,0.532094,0.531498,0.791286,3.791500,83.080000,5.275000


Training completed!


In [9]:
# Оценка на валидации

# Eval
eval_results = trainer.evaluate(val_dataset)
print("Validation results:")
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

Validation results:
eval_loss: 0.5259
eval_log_loss: 0.5256
eval_auc: 0.7777
eval_runtime: 3.7299
eval_samples_per_second: 84.4530
eval_steps_per_second: 5.3620
epoch: 3.0000


In [10]:
# Предсказания и генерация submission

# Предсказания на test
test_predictions = trainer.predict(test_dataset)
test_probs = np.exp(test_predictions.predictions) / np.sum(np.exp(test_predictions.predictions), axis=1)[:, None]

# Submission (Kaggle подхватит автоматически)
submission = pd.DataFrame({
    'ID': test_df['ID'],
    'is_bot': test_probs[:, 1]  # Вероятность бота
})
submission.to_csv('submission.csv', index=False)
print("Submission saved! Shape:", submission.shape)
print(submission.head())

Submission saved! Shape: (676, 2)
                                   ID    is_bot
0  af36ac2aa9734738bbd533db8e5fb43a_0  0.273325
1  af36ac2aa9734738bbd533db8e5fb43a_1  0.449965
2  cdc2c5c605144c8e8dd5e9ea3d1352fc_0  0.087444
3  cdc2c5c605144c8e8dd5e9ea3d1352fc_1  0.119454
4  ed19efdedcb24600aea67c968aba5520_0  0.339512
